In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score

### Exploratory Data Analysis (EDA)

In [4]:
df = pd.read_csv('/home/fatih/Documents/Projects/penggalian-data/modul4/dataset/supermarket_sales.csv')

print("\n--- Descriptive Statistics ---")
print(df.describe())

print("\n--- Check for Missing Values ---")
print(df.isnull().sum())

# Preprocessing categorical variables
label_encoders = {}
for column in df.select_dtypes(include=['object']).columns:
    if column not in ['Invoice ID', 'Date', 'Time']: # skip irrelevant identifiers
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])
        label_encoders[column] = le

numeric_df = df.select_dtypes(include=[np.number])

# Visualizing target variable
plt.figure(figsize=(8, 5))
sns.histplot(df['Rating'], kde=True, bins=20)
plt.title('Distribution of Target Variable (Rating)')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.savefig('/home/fatih/Documents/Projects/penggalian-data/modul4/rating_distribution.png')
plt.close()

# Correlation Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.savefig('/home/fatih/Documents/Projects/penggalian-data/modul4/correlation_heatmap.png')
plt.close()


--- Descriptive Statistics ---
        Unit price     Quantity       Tax 5%        Total        cogs  \
count  1000.000000  1000.000000  1000.000000  1000.000000  1000.00000   
mean     55.672130     5.510000    15.379369   322.966749   307.58738   
std      26.494628     2.923431    11.708825   245.885335   234.17651   
min      10.080000     1.000000     0.508500    10.678500    10.17000   
25%      32.875000     3.000000     5.924875   124.422375   118.49750   
50%      55.230000     5.000000    12.088000   253.848000   241.76000   
75%      77.935000     8.000000    22.445250   471.350250   448.90500   
max      99.960000    10.000000    49.650000  1042.650000   993.00000   

       gross margin percentage  gross income      Rating  
count              1000.000000   1000.000000  1000.00000  
mean                  4.761905     15.379369     6.97270  
std                   0.000000     11.708825     1.71858  
min                   4.761905      0.508500     4.00000  
25%            

/tmp/ipykernel_166844/2075970373.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df.select_dtypes(include=['object']).columns:


### Feature Selection using RFE

In [6]:
X = numeric_df.drop(['Rating'], axis=1)
y = numeric_df['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Correlation-based selection (Select features with highest correlation to Rating)
correlations = numeric_df.corr()['Rating'].drop('Rating').abs()
corr_selected_features = correlations[correlations > 0.05].index.tolist()
print(f"Features selected by correlation (>0.05): {corr_selected_features}")

# RFE
model = RandomForestRegressor(random_state=42)
rfe = RFE(estimator=model, n_features_to_select=5)
rfe.fit(X_train_scaled, y_train)

rfe_selected_features = X.columns[rfe.support_].tolist()
print(f"Features selected by RFE: {rfe_selected_features}")

# Feature importance visualization
rfe_ranking = pd.DataFrame({'Feature': X.columns, 'Ranking': rfe.ranking_}).sort_values('Ranking')
plt.figure(figsize=(10, 6))
sns.barplot(x='Ranking', y='Feature', data=rfe_ranking)
plt.title('RFE Feature Ranking (1 is best)')
plt.savefig('/home/fatih/Documents/Projects/penggalian-data/modul4/rfe_ranking.png')
plt.close()

Features selected by correlation (>0.05): []
Features selected by RFE: ['Product line', 'Unit price', 'Tax 5%', 'cogs', 'gross income']


### Dimensional Reduction

In [7]:
# 3. Dimensionality Reduction (PCA)
pca = PCA()
pca.fit(X_train_scaled)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), marker='o', linestyle='--')
plt.title('Explained Variance by Number of Principal Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.axhline(y=0.90, color='r', linestyle='-')
plt.text(0.5, 0.85, '90% threshold', color = 'red', fontsize=10)
plt.savefig('/home/fatih/Documents/Projects/penggalian-data/modul4/pca_variance.png')
plt.close()

# Let's say we choose 6 components based on the plot
n_components = 6
pca_optimal = PCA(n_components=n_components)
X_train_pca = pca_optimal.fit_transform(X_train_scaled)
X_test_pca = pca_optimal.transform(X_test_scaled)

### Evaluation Model

In [8]:
# 1. Baseline Model (All Numeric Features)
baseline_model = RandomForestRegressor(random_state=42)
baseline_model.fit(X_train_scaled, y_train)
y_pred_base = baseline_model.predict(X_test_scaled)
r2_base = r2_score(y_test, y_pred_base)

# 2. RFE Selected Features Model
X_train_rfe = rfe.transform(X_train_scaled)
X_test_rfe = rfe.transform(X_test_scaled)
rfe_model = RandomForestRegressor(random_state=42)
rfe_model.fit(X_train_rfe, y_train)
y_pred_rfe = rfe_model.predict(X_test_rfe)
r2_rfe = r2_score(y_test, y_pred_rfe)

# 3. PCA Model
pca_model = RandomForestRegressor(random_state=42)
pca_model.fit(X_train_pca, y_train)
y_pred_pca = pca_model.predict(X_test_pca)
r2_pca = r2_score(y_test, y_pred_pca)

print(f"R^2 Score (All Features): {r2_base:.4f}")
print(f"R^2 Score (RFE Selected): {r2_rfe:.4f}")
print(f"R^2 Score (PCA {n_components} Components): {r2_pca:.4f}")

results = pd.DataFrame({
    'Model Approach': ['All Features', 'RFE Selection', f'PCA ({n_components} components)'],
    'R2 Score (Accuracy)': [r2_base, r2_rfe, r2_pca]
})

plt.figure(figsize=(8, 5))
sns.barplot(x='Model Approach', y='R2 Score (Accuracy)', data=results)
plt.title('Model Performance Comparison (R2 Score)')
plt.ylabel('R2 Score (Higher is better)')
plt.ylim(min(r2_base, r2_rfe, r2_pca) - 0.05, max(r2_base, r2_rfe, r2_pca) + 0.05)
plt.savefig('/home/fatih/Documents/Projects/penggalian-data/modul4/evaluation_comparison.png')
plt.close()


R^2 Score (All Features): -0.1398
R^2 Score (RFE Selected): -0.1405
R^2 Score (PCA 6 Components): -0.1390
